In [1]:
import sqlite3
import random
import time
import requests
import pandas as pd
random.seed(42)

In [2]:
conn = sqlite3.connect("school.db")
cur = conn.cursor()
cur.execute("""
CREATE TABLE students (
student_id INTEGER PRIMARY KEY,
name TEXT, major TEXT, gpa REAL
)""")
cur.execute("""
CREATE TABLE enrollments (
enrollment_id INTEGER PRIMARY KEY,
student_id INTEGER, course TEXT, grade TEXT,
FOREIGN KEY (student_id) REFERENCES students(student_id)
)""")
# ... populate with sample rows (full code in the demo notebook) ...
conn.commit()

In [3]:
query = """
SELECT name, gpa
FROM students
WHERE major = 'CS' AND gpa >= 3.5
ORDER BY gpa DESC

"""
print(pd.read_sql(query, conn))

Empty DataFrame
Columns: [name, gpa]
Index: []


In [4]:
query = """
SELECT students.name, enrollments.course, enrollments.grade
FROM students
INNER JOIN enrollments ON students.student_id = enrollments.student_id
"""
print(pd.read_sql(query, conn))

Empty DataFrame
Columns: [name, course, grade]
Index: []


In [5]:
query = """
SELECT major, COUNT(*) AS n_students, AVG(gpa) AS avg_gpa
FROM students
GROUP BY major
HAVING COUNT(*) > 2
ORDER BY avg_gpa DESC
"""
print(pd.read_sql(query, conn))

Empty DataFrame
Columns: [major, n_students, avg_gpa]
Index: []


In [6]:
query = """
SELECT major, AVG(gpa) AS avg_gpa
FROM students
GROUP BY major
"""
df = pd.read_sql(query, conn)
print(df)
conn.close() # same discipline as closing a file

Empty DataFrame
Columns: [major, avg_gpa]
Index: []


In [8]:
import requests

url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 37.5665,      # Seoul
    "longitude": 126.9780,
    "current_weather": True,
}

try:
    response = requests.get(url, params=params, timeout=10)
    print("status code:", response.status_code)
    
    data = response.json()
    print(data["current_weather"])

except requests.exceptions.RequestException:
    # Falls back to sample data if there is no Wi-Fi
    data = {
        "current_weather": {
            "temperature": 18.4,
            "windspeed": 9.1
        }
    }
    
    print("No internet connection. Using sample weather data.")
    print(data["current_weather"])

status code: 200
{'time': '2026-10-08T05:00', 'interval': 900, 'temperature': 22.3, 'windspeed': 2.3, 'winddirection': 252, 'is_day': 1, 'weathercode': 0}


In [9]:
weather_df = pd.DataFrame([data["current_weather"]])
print(weather_df)

               time  interval  temperature  windspeed  winddirection  is_day  \
0  2026-10-08T05:00       900         22.3        2.3            252       1   

   weathercode  
0            0  


In [10]:
from bs4 import BeautifulSoup
url = "http://books.toscrape.com"
headers = {"User-Agent": "KDU-DataScience-Student-Demo"}
response = requests.get(url, headers=headers, timeout=10)
soup = BeautifulSoup(response.text, "html.parser")
titles = [h.get_text(strip=True) for h in soup.select("h3 a")]
print(titles[:5])
time.sleep(1) # be polite before any next request

['A Light in the ...', 'Tipping the Velvet', 'Soumission', 'Sharp Objects', 'Sapiens: A Brief History ...']
